## This is the working jupyter notebook for the Quant Bio Lab Project. 

#### First Steps
1) Input Image
2) Apply Gaussian Filter for blurring
3) Apply Thresholding via CV2 Library
    - Do I need to normalize the stack? 
4) Apply Contour finding via CV2 Library
5) Apply to the whole image stack
6) Pray. 

In [9]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def gaussian_kernal(size,std):
    kernel = np.fromfunction(
        lambda x,y: np.divide(1,2*np.pi* std**2) * 
        np.exp(
            -((x-(size-1)/2)**2 + (y-(size-1)/2)**2) / (2* std**2)
               ),
        (size,size)
    )
    return np.array(kernel/np.sum(kernel))

def find(frame,mask,contour,pixelSize):

    """
    This function takes in an image and draws the given contours.  It returns an inverted boolean array
    where true = pixel within a contour. This gives the number of pixels within a given contour to calculate area, as well as the 
    starting point for blob labeling and tracking.
    """
    #dont want to edit the original image
    filled_mask = np.copy(mask)
    #draw the contour
    filled_mask = cv.drawContours(filled_mask,[contour], 0,(0, 255, 0),thickness=cv.FILLED)
    #boolean mask and returning it, ensuring true = 0, false = 1. The sum of this is pixel size. 
    image_mask = filled_mask > 0
    total_pixels = np.sum(~image_mask.astype(int))
    image_mask = ~image_mask
    total_pixels = np.sum(image_mask.astype(int))
    #output [frame, x,y, Area, TotalIntensity]
    total_intensity = mask*(image_mask.astype(int))
    ##Find X,Y center of mass based on contours. 
    #https://docs.opencv.org/3.4/dd/d49/tutorial_py_contour_features.html
    M = cv.moments(contour)
    x = int(M['m10']/M['m00'])
    y = int(M['m01']/M['m00'])
    #in units of pixelSize(micron squared)
    total_area = total_pixels*(pixelSize**2)
    return [frame,[x,y],total_area,total_intensity]

## What do I want to pull from individual contours?
Num Pixels = Area Calculations.
Centers of Blobs = Pixels Coordinates in Frame,X,Y

In [10]:
kernel = gaussian_kernal(3,np.sqrt(3))
low_thresh: int = 13
high_thresh: int = 50
filter_val: int = 10
pixel_size: float = 0.115

## What is happening here?

In this cell, we are iterating through the tiff stack, tresholding it, finding contours and subsequently identifying the areas within each contour, its centroid, and also filtering bad contours. 

In [11]:
import skimage.io as io
tiff_stack = io.imread("/Users/cmdb/Quant_Bio_Project/Quant-Bio-Project/cell2.tif",plugin='tifffile')
tiff_stack = (tiff_stack/256).astype(np.uint8)
copied = np.copy(tiff_stack)
frame_dict = {}
for i in range(len(tiff_stack)):
    ret,thresh = cv.threshold(tiff_stack[i,:,:],low_thresh,high_thresh,cv.THRESH_BINARY)
    contours,hierarchy = cv.findContours(thresh, 1, 2)
    #filtering out contours less than 10
    contours = [lst for lst in contours if len(lst) >= filter_val]
    frame_dict[f"Timepoint {i}"] = []
    for j in contours:
        #thresholding step needed to skip bad areas. Area == 0 is nothing. 
        area = cv.moments(j)["m00"]
        if area > 0:
            centroid_output = find(i,copied[i,:,:],j,pixel_size)
            frame_dict[f"Timepoint {i}"].append(centroid_output)


## Whats going on here?

Here we are applying a simple tracker by assigning a blob in n to presumably the same blob in n+1 by eulucidian distance. This isn't the ideal method to assign paths, however our images are simple. 

In [18]:
##Challenges
# iterate through the dictionary by frame ##### success
# iterate through the dictionary by frame and frame + 1 ##### success
# pull x,y coords

#keys list
list_of_keys = frame_dict.keys()
for i in range(len(list_of_keys)-1):
    print(i)
    print(i+1)
    print(" ")

0
1
 
1
2
 
2
3
 
3
4
 
4
5
 
5
6
 
6
7
 
7
8
 
8
9
 
9
10
 
10
11
 
11
12
 
12
13
 
13
14
 
14
15
 
15
16
 
16
17
 
17
18
 
18
19
 
19
20
 
20
21
 
21
22
 
22
23
 
23
24
 
24
25
 
25
26
 
26
27
 
27
28
 
28
29
 
29
30
 
30
31
 
31
32
 
32
33
 
33
34
 
34
35
 
35
36
 
36
37
 
37
38
 
38
39
 
39
40
 
40
41
 
41
42
 
42
43
 
43
44
 
44
45
 
45
46
 
46
47
 
47
48
 
48
49
 
49
50
 
50
51
 
51
52
 
52
53
 
53
54
 
54
55
 
55
56
 
56
57
 
57
58
 
58
59
 
59
60
 
60
61
 
61
62
 
62
63
 
63
64
 
64
65
 
65
66
 
66
67
 
67
68
 
68
69
 
69
70
 
70
71
 
71
72
 
72
73
 
73
74
 
74
75
 
75
76
 
76
77
 
77
78
 
78
79
 
79
80
 
80
81
 
81
82
 
82
83
 
83
84
 
84
85
 
85
86
 
86
87
 
87
88
 
88
89
 
89
90
 
90
91
 
91
92
 
92
93
 
93
94
 
94
95
 
95
96
 
96
97
 
97
98
 
98
99
 
99
100
 
100
101
 
101
102
 
102
103
 
103
104
 
104
105
 
105
106
 
106
107
 
107
108
 
108
109
 
109
110
 
110
111
 
111
112
 
112
113
 
113
114
 
114
115
 
115
116
 
116
117
 
117
118
 
118
119
 
119
120
 
120
121
 
121
122


## previous work

In [12]:

"""
    
"""
# image = cv.filter2D(tiff_stack[150,:,:],-1,kernel)
# ret,thresh = cv.threshold(image,low_thresh,high_thresh,cv.THRESH_BINARY)
# contours,hierarchy = cv.findContours(thresh, 1, 2)
# #filtering out contours less than whatever pixels in size
# contours = [lst for lst in contours if len(lst) >= filter_val]

# test = cv.drawContours(image, contours, -5, (0, 255, 0), 1) 
# plt.imshow(test)







'\n    \n'